In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.linear_model import Ridge, Perceptron, BayesianRidge, ElasticNet, Lasso
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_extraction.text import CountVectorizer
import sklearn.ensemble as ensemble
from sklearn.base import TransformerMixin
from sklearn.multioutput import MultiOutputRegressor

# https://towardsdatascience.com/preprocessing-textual-data-c0527ef8b8c5
# https://www.geeksforgeeks.org/removing-stop-words-nltk-python/
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

## loading data and preprocessing

First we need to load from CSV and inspect the data.

In [ ]:
def round_to_nearest_half(array):
    
    # algorithm: round(x * 2) / 2
    return np.divide(np.round(np.multiply(array, 2)), 2)

In [ ]:
train_df = pd.read_csv('/kaggle/input/feedback-prize-english-language-learning/train.csv')
test_df = pd.read_csv('/kaggle/input/feedback-prize-english-language-learning/test.csv')

def clean_row(row):
    # filter out empty strings, lowercase all words, and remove stop words
    stop_words = set(stopwords.words('english'))
    row['full_text'] = list(
    map(lambda word : word.lower().strip(),
        filter(lambda word :
               not word.lower() in stop_words
               or word == '',
               word_tokenize(row['full_text']))))

    return row


def clean_data(train, test):
    return train.apply(clean_row, axis=1, result_type='broadcast'), test.apply(clean_row, axis=1, result_type='broadcast')


# train_df, test_df = clean_data(train_df, test_df)
# train_df

# split data into X_train and y_train
X_train = train_df['full_text']
y_train = train_df.drop(['full_text', 'text_id'], axis=1)

# for bayesian ridge
class DenseTransformer(TransformerMixin):

    def fit(self, X, y=None, **fit_params):
        return self

    def transform(self, X, y=None, **fit_params):
        return X.todense()

# create the count vectorizer, model, and pipeline
countvec = CountVectorizer(stop_words="english")
# model = Ridge(alpha=10000)
# model = MultiOutputRegressor(LinearSVC(C=0.00005))
# model = DecisionTreeRegressor(max_depth = 3)

### for models which give an error like 'this model expects a 1d array...' we can fix the model using
### MultiOutputRegressor. this effectively applies the model to each output column separately
# model = MultiOutputRegressor(ensemble.GradientBoostingRegressor(learning_rate=0.1))

# model = MultiOutputRegressor(Perceptron())

# model = BayesianRidge()

# model = ElasticNet(alpha=0.05, max_iter=1000, l1_ratio=0.1)

model = Lasso(alpha=0.01, tol=1e-5)

pipe = make_pipeline(countvec, model)

cross_val_results = pd.DataFrame(
    ### for reasons unknown to me, we need to use integers for the MultiOutputRegressor to work right.
    ### because the output is multiples of 0.5, we just double it then convert to int
    ### see https://stackoverflow.com/questions/45346550/valueerror-unknown-label-type-unknown
    cross_validate(pipe, X_train, np.multiply(y_train, 2).astype('int'), return_train_score=True)
)

# report performance
print(cross_val_results.mean())

# y_train.astype('int')

Now we need to export this dataframe to csv.

Use the model and get results.

In [ ]:
# pipe.fit(X_train, y_train)
### multiply output by 2 here as well
pipe.fit(X_train, np.multiply(y_train, 2).astype('int'))
result_vals = pipe.predict(test_df['full_text'])

rounded_result_vals = round_to_nearest_half(result_vals)
results = pd.DataFrame(result_vals / 2.0, ### cut the values in half again
                       columns=['cohesion', 'syntax', 'vocabulary', 'phraseology', 'grammar', 'conventions'])
results.insert(0, 'text_id', test_df['text_id'])
print(results)
results.to_csv('/kaggle/working/submission.csv', index=False)